# Soccer Scouting Platform — Tactical Analysis

Opposition scouting notebook built on **StatsBomb Open Data** (Premier League 2015/16).

**Metrics covered:** xG profile, PPDA (pressing intensity), shot maps, attack zones.

---

### Data attribution (required by StatsBomb)

Event data sourced from [StatsBomb Open Data](https://github.com/statsbomb/open-data).
When sharing this notebook publicly, include the StatsBomb logo from their [media pack](https://statsbomb.com/media-pack/).

**Settings:** Enable **Internet** in the notebook settings before running.

In [ ]:
!pip install -q mplsoccer sqlalchemy requests

In [ ]:
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from mplsoccer import Pitch
from sqlalchemy import create_engine, text

STATSBOMB_BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
COMPETITION_ID = 2
SEASON_ID = 27
TARGET_TEAM = "Chelsea"
DATA_DIR = Path("/kaggle/working/statsbomb")
DB_PATH = Path("/kaggle/working/scouting.db")

DEFENSIVE_ACTIONS = ("Pressure", "Foul Committed", "Tackle", "Interception", "Block")

def fetch_json(url: str):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()

def save_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload), encoding="utf-8")

print("Config ready.")

In [ ]:
matches_path = DATA_DIR / "matches" / str(COMPETITION_ID) / f"{SEASON_ID}.json"
if not matches_path.exists():
    save_json(matches_path, fetch_json(f"{STATSBOMB_BASE}/matches/{COMPETITION_ID}/{SEASON_ID}.json"))

matches = json.loads(matches_path.read_text(encoding="utf-8"))
team_lower = TARGET_TEAM.lower()
team_matches = [
    match for match in matches
    if team_lower in (
        match["home_team"]["home_team_name"].lower(),
        match["away_team"]["away_team_name"].lower(),
    )
]
print(f"Downloading events for {len(team_matches)} {TARGET_TEAM} matches...")

for index, match in enumerate(team_matches, start=1):
    match_id = match["match_id"]
    event_path = DATA_DIR / "events" / f"{match_id}.json"
    if not event_path.exists():
        save_json(event_path, fetch_json(f"{STATSBOMB_BASE}/events/{match_id}.json"))
        time.sleep(0.05)
    if index % 10 == 0:
        print(f"  {index}/{len(team_matches)} done")

print("Download complete.")

In [ ]:
def flatten_location(location, index):
    if not location or len(location) <= index:
        return None
    return float(location[index])

def event_rows(match_id, events):
    rows = []
    for event in events:
        team = event.get("team") or {}
        player = event.get("player") or {}
        event_type = event.get("type") or {}
        shot = event.get("shot") or {}
        pass_data = event.get("pass") or {}
        location = event.get("location")
        end_location = pass_data.get("end_location") or shot.get("end_location")
        rows.append(
            {
                "event_id": f"{match_id}_{event['index']}",
                "match_id": match_id,
                "team_name": team.get("name"),
                "player_name": player.get("name"),
                "event_type": event_type.get("name"),
                "minute": event.get("minute"),
                "location_x": flatten_location(location, 0),
                "location_y": flatten_location(location, 1),
                "is_goal": int(shot.get("outcome", {}).get("name") == "Goal"),
                "is_shot": int(event_type.get("name") == "Shot"),
                "shot_xg": shot.get("statsbomb_xg"),
                "shot_type": (shot.get("type") or {}).get("name"),
                "play_pattern": (event.get("play_pattern") or {}).get("name"),
            }
        )
    return rows

all_events = []
for event_file in sorted((DATA_DIR / "events").glob("*.json")):
    events = json.loads(event_file.read_text(encoding="utf-8"))
    all_events.extend(event_rows(int(event_file.stem), events))

events_df = pd.DataFrame(all_events)
engine = create_engine(f"sqlite:///{DB_PATH}")
events_df.to_sql("events", engine, if_exists="replace", index=False)
print(f"Loaded {len(events_df):,} events into SQLite.")

In [ ]:
shots = pd.read_sql(
    """
    SELECT *
    FROM events
    WHERE team_name = :team
      AND is_shot = 1
      AND location_x IS NOT NULL
      AND location_y IS NOT NULL
    """,
    engine,
    params={"team": TARGET_TEAM},
)

goals = int(shots["is_goal"].sum())
total_xg = float(shots["shot_xg"].fillna(0).sum())
print(f"{TARGET_TEAM} — {len(shots)} shots, {goals} goals, {total_xg:.2f} xG ({goals - total_xg:+.2f} difference)")

In [ ]:
def pitch_zone(y):
    if y < 80 / 3:
        return "Left"
    if y > (80 / 3) * 2:
        return "Right"
    return "Central"

shots["zone"] = shots["location_y"].apply(pitch_zone)
zone_df = (
    shots.groupby("zone")
    .agg(shots=("event_id", "count"), goals=("is_goal", "sum"), total_xg=("shot_xg", "sum"))
    .reset_index()
)
zone_df

In [ ]:
def ppda_for_match(match_events, team):
    team_actions = match_events[
        (match_events["team_name"] == team) & (match_events["event_type"].isin(DEFENSIVE_ACTIONS))
    ]
    opponent_passes = match_events[
        (match_events["team_name"] != team) & (match_events["event_type"] == "Pass")
    ]
    if team_actions.empty:
        return None
    return len(opponent_passes) / len(team_actions)

ppda_values = []
for match_id in shots["match_id"].unique():
    match_events = pd.read_sql(
        "SELECT team_name, event_type FROM events WHERE match_id = :match_id",
        engine,
        params={"match_id": int(match_id)},
    )
    value = ppda_for_match(match_events, TARGET_TEAM)
    if value is not None:
        ppda_values.append(value)

ppda = round(sum(ppda_values) / len(ppda_values), 2)
print(f"{TARGET_TEAM} PPDA: {ppda} (lower = more aggressive press)")

In [ ]:
pitch = Pitch(pitch_type="statsbomb", pitch_color="#22312b", line_color="#c7d5cc")
fig, ax = pitch.draw(figsize=(12, 8))

goal_shots = shots[shots["is_goal"] == 1]
other_shots = shots[shots["is_goal"] == 0]

pitch.hexbin(
    other_shots["location_x"], other_shots["location_y"],
    ax=ax, gridsize=(18, 12), cmap="YlOrRd", alpha=0.45, mincnt=1,
)
pitch.scatter(other_shots["location_x"], other_shots["location_y"], ax=ax, s=18, c="white", alpha=0.35)
if not goal_shots.empty:
    pitch.scatter(
        goal_shots["location_x"], goal_shots["location_y"],
        ax=ax, s=160, marker="*", c="#4fd1c5", edgecolors="white",
    )

ax.set_title(f"{TARGET_TEAM} Shot Map — Premier League 2015/16", color="white", fontsize=14)
fig.patch.set_facecolor("#1a1a1a")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
colors = ["#034694", "#6cabdd", "#4fd1c5"]
axes[0].bar(zone_df["zone"], zone_df["shots"], color=colors)
axes[0].set_title("Shots by Zone")
axes[1].bar(zone_df["zone"], zone_df["total_xg"], color=colors)
axes[1].set_title("xG by Zone")
fig.suptitle(f"{TARGET_TEAM} Attack Zones", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Next steps

- Change `TARGET_TEAM` to analyze another Premier League club.
- Full interactive dashboard: deploy the GitHub repo to **Streamlit Community Cloud**.
- Link this notebook on your Kaggle profile and GitHub README.